# WignerFlow: stable argon fine-tune and v0 comparison (October 30)

**What this does.** The full-data argon run (`WignerFlow_runs/argon_full_oct29/train`) was stable through the m=2 phase (step 20000: no crash, val RDF L1 0.433) but diverged when the unroll length jumped 2 → 4 at step 20050 (train loss 0.12 → 11–26; validation at step 22000: 4/4 rollouts crashed after 272 steps). This notebook does not restart from scratch:

1. ranks the old run's best non-crashing snapshots with longer (8 × 2000-step) validation rollouts and picks the most stable one;
2. fine-tunes it with `configs/argon/B_S3a_stable_ft.yaml`: unroll 2 → 3 → 4 with an LR re-warmup after each switch, loss on every unroll step, a loss-spike guard, grad clip 0.5, LR 1e-4, the repulsion prior, noise annealing and an overlap penalty; checkpoints are selected by stability (`best_stable.pt`);
3. profiles training (throughput, step-time breakdown, peak memory, GPU utilization, torch.profiler trace) into `<FT>/profiling/`;
4. evaluates v0 and `best_stable.pt` on the held-out test trajectory (8 starts × 3 seeds × 1000 steps) and writes the four-metric verdict (temperature, RDF, VACF, diffusion errors) to `<FT>/eval_multiseed/v0_vs_v1.json`.

**How to run.** Runtime > Change runtime type > A100 GPU. Leave the form fields at their defaults and use Runtime > Run all. If the session disconnects, run all again: ranking, training (from `ckpt.pt`), profiling and evaluation each resume or skip finished parts.

Everything is written under `MyDrive/WignerFlow_runs/<RUN_NAME>/<FT_NAME>/`.

In [ ]:
# ---- Form fields (edit here only) ----
RUN_NAME = 'argon_full_oct29'  # @param {type:"string"}
FT_NAME = 'train_stable_ft'  # @param {type:"string"}
BRANCH = 'feat/argon-stable-oct30'  # @param {type:"string"}
INIT_SNAPSHOT = 'auto'  # @param {type:"string"}
RANK_TOP = 3  # @param {type:"integer"}
RANK_STEPS = 2000  # @param {type:"integer"}
EVAL_STARTS = 8  # @param {type:"integer"}
EVAL_SEEDS = 3  # @param {type:"integer"}
EVAL_STEPS = 1000  # @param {type:"integer"}
# INIT_SNAPSHOT: 'auto' = most stable of the old run's snapshots; or a file name such as 'step016000.pt'
# --------------------------------------
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, time, shutil
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/WignerFlow_runs') / RUN_NAME
FT = RUN / FT_NAME
FT.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/WignerFlow')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ.pop('PYTHONPATH', None)

def run(command, log_name=None):
    print(' '.join(map(str, command)), flush=True)
    start = time.perf_counter()
    log = open(FT / log_name, 'a') if log_name else None
    try:
        with subprocess.Popen(list(map(str, command)), cwd=REPO if REPO.exists() else '/content',
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='')
                if log:
                    log.write(line); log.flush()
            status = proc.wait()
        with open(FT / 'commands.jsonl', 'a') as f:
            f.write(json.dumps(dict(command=list(map(str, command)), wall_s=time.perf_counter()-start, exit=status))+'\n')
        if status:
            raise RuntimeError(f'Command failed with exit {status}')
    finally:
        if log:
            log.close()

if not REPO.exists():
    run(['git', 'clone', 'https://github.com/yaniguan/WignerFlow.git', REPO])
run(['git', 'fetch', 'origin', BRANCH])
run(['git', 'checkout', '-B', BRANCH, 'origin/' + BRANCH])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[simulation,train,test]'], 'install.log')
run(['git', 'rev-parse', 'HEAD'], 'source_commit.txt')
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > A100 GPU, then rerun.'
torch.set_num_threads(2)
print(torch.cuda.get_device_name(0))
(FT / 'gpu.json').write_text(json.dumps(dict(device=torch.cuda.get_device_name(0), torch=torch.__version__), indent=2))
run([sys.executable, '-m', 'pytest', '-q', 'tests/test_finetune.py', 'tests/test_stability.py'], 'pytest.log')

## Stage the full argon data (already on Drive)

Copies the ten 0.5 ns trajectories from `<RUN_NAME>/data/argon_full` to local disk. If any are missing it regenerates them with OpenMM (slow; normally skipped).

In [ ]:
import h5py, yaml
DATA_DRIVE = RUN / 'data' / 'argon_full'
DATA_LOCAL = REPO / 'data' / 'argon_full'
DATA_LOCAL.mkdir(parents=True, exist_ok=True)
def complete(p):
    try:
        with h5py.File(p, 'r') as h:
            return bool(h.attrs['complete']) and len(h['positions']) == 100000
    except Exception:
        return False
if not all(complete(DATA_DRIVE / f'traj_{i:03d}.h5') for i in range(10)):
    run([sys.executable, '-m', 'pip', 'install', '-q', 'openmm[cuda12]>=8.1'], 'install.log')
    run([sys.executable, 'scripts/gen_data.py', '--config', 'configs/data/argon_full.yaml',
         '--platform', 'CUDA', '--out_dir', DATA_DRIVE], 'generation.log')
for i in range(10):
    src = DATA_DRIVE / f'traj_{i:03d}.h5'
    assert complete(src), src
    dst = DATA_LOCAL / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
shutil.copy2(DATA_DRIVE / 'config.yaml', DATA_LOCAL / 'config.yaml')
print('data ready:', DATA_LOCAL)

## Pick the starting snapshot and write the runtime config

`init_rank.json` stores the ranking; it is reused on reruns. Ranking takes about 5–10 minutes per candidate on an A100.

In [ ]:
OLD = RUN / 'train'
cfg = yaml.safe_load((REPO / 'configs/argon/B_S3a_stable_ft.yaml').read_text())
cfg['data_dir'] = str(DATA_LOCAL)
cfg['out_dir'] = str(FT)
PRE_CONFIG = FT / 'rank_config.yaml'
PRE_CONFIG.write_text(yaml.safe_dump(cfg))
RANK = FT / 'init_rank.json'
CONFIG = FT / 'B_S3a_stable_ft_runtime.yaml'
if CONFIG.exists():
    cfg = yaml.safe_load(CONFIG.read_text())   # resume: keep the original choice
    INIT = cfg['train']['init_from']
elif INIT_SNAPSHOT != 'auto':
    INIT = str(OLD / 'snapshots' / INIT_SNAPSHOT)
else:
    if not RANK.exists():
        run([sys.executable, 'scripts/rank_snapshots.py', '--config', PRE_CONFIG, '--run_dir', OLD,
             '--top', RANK_TOP, '--starts', 8, '--steps', RANK_STEPS, '--sampling_steps', 10,
             '--device', 'cuda', '--out', RANK], 'rank.log')
    INIT = json.loads(RANK.read_text())['chosen']
assert Path(INIT).exists(), INIT
cfg['train']['init_from'] = str(INIT)
if not CONFIG.exists():
    CONFIG.write_text(yaml.safe_dump(cfg))
print('init_from:', INIT)
print('config:', CONFIG)
if RANK.exists():
    for r in json.loads(RANK.read_text())['ranking']:
        print(Path(r['snapshot']).name, 'crash_frac', r['val_crash_frac'], 'steps', r['val_steps_before_crash'],
              'rdf_l1', r.get('val_rdf_l1'), 'T_end', r.get('val_T_end'))

## Profile (once) and train

Profiling runs once before training (30 training steps at unroll m=4 with a CUDA-synchronized phase breakdown, a short torch.profiler trace, 50 rollout steps, nvidia-smi sampling). During training, GPU utilization and memory are sampled every 5 s into `profiling/train_gpu_samples.csv`, and training throughput is computed from the training log. Everything is merged into `profiling/summary.json`.

Training: 16,000 steps (unroll 2 → 3 at 3000 → 4 at 8000), validation every 2000 steps. Rerun this cell to resume from `ckpt.pt`.

In [ ]:
from equitraj.profiling import GpuSampler, write_summary
PROF = FT / 'profiling'
if not (PROF / 'trace.json').exists():
    run([sys.executable, 'scripts/profile_run.py', '--config', CONFIG, '--ckpt', INIT, '--m', 4,
         '--train-steps', 30, '--rollout-steps', 50, '--rollout-batch', 1, '--profile', 'train',
         '--gpu-interval', 1.0, '--out', PROF], 'profile.log')
torch.cuda.empty_cache()
if not (FT / 'done.json').exists():
    t0 = time.time()
    sampler = GpuSampler(str(PROF / f'train_gpu_samples_{int(t0)}.csv'), 5.0, timeout_s=15.0).start()
    try:
        run([sys.executable, '-m', 'equitraj.train', '--config', CONFIG, '--device', 'cuda'], 'training.log')
    finally:
        sampler.stop()
        write_summary(PROF, {f'training_session_{int(t0)}': {'wall_s': time.time() - t0, 'gpu': sampler.summary()}})
# training throughput per unroll length from the training log
rows = [json.loads(l) for l in (FT / 'training.log').read_text().splitlines() if l.startswith('{"step"') and '"loss"' in l]
thr = {}
for a, b in zip(rows, rows[1:]):
    if b['m'] == a['m'] and b['elapsed_h'] > a['elapsed_h'] and b['step'] - a['step'] == 50:
        thr.setdefault(f"m{b['m']}", []).append(50 / ((b['elapsed_h'] - a['elapsed_h']) * 3600))
import numpy as np
write_summary(PROF, {'training_throughput_steps_per_s': {k: float(np.median(v)) for k, v in thr.items()},
                     'training_samples_per_s': {k: float(np.median(v)) * cfg['train']['batch_size'] for k, v in thr.items()},
                     'skipped_updates_total': rows[-1].get('skipped_updates') if rows else None})
print(json.dumps(json.loads((PROF / 'summary.json').read_text()).get('training_throughput_steps_per_s'), indent=2))
for l in (FT / 'log.jsonl').read_text().splitlines():
    m = json.loads(l)
    print(m['step'], 'crash', m['val_crash_frac'], 'sel_crash', m.get('sel_val_crash_frac'), 'rdf', m.get('val_rdf_l1'),
          'T_end', m.get('val_T_end'), 'stability', m.get('stability_score'))

## Evaluate v0 vs the fine-tuned best_stable.pt on the held-out test trajectory

8 starts × 3 seeds × 1000 steps, 10 flow steps, model-only dynamics. `v0_vs_v1.json` states whether v1 beats v0 on all four of temperature, RDF, VACF and diffusion errors (lower is better) without more crashes.

In [ ]:
EVAL = FT / 'eval_multiseed'
CK = FT / 'checkpoints'
CK.mkdir(exist_ok=True)
V1 = CK / 'argon_flow_v1_stable_ft.pt'
if not V1.exists():
    shutil.copy2(FT / 'best_stable.pt', V1)
BASELINE = CK / 'argon_flow_v0.pt'
shutil.copy2(REPO / 'checkpoints/argon_flow_v0.pt', BASELINE)
if not (EVAL / 'metrics.json').exists() or len(json.loads((EVAL / 'metrics.json').read_text())['checkpoints']) < 2:
    run([sys.executable, 'scripts/eval_multiseed.py', '--checkpoints', BASELINE, V1,
         '--data_dir', DATA_LOCAL, '--starts', EVAL_STARTS, '--seeds', EVAL_SEEDS, '--steps', EVAL_STEPS,
         '--sampling_steps', 10, '--device', 'cuda', '--threads', 2, '--out', EVAL], 'evaluation.log')
run([sys.executable, 'scripts/compare_v0.py', '--metrics', EVAL / 'metrics.json', '--out', EVAL / 'v0_vs_v1.json'])
try:
    run([sys.executable, 'scripts/export_ckpt.py', '--run', FT, '--ckpt', 'best_stable.pt',
         '--out', CK / 'argon_flow_v1_stable_ft_release.pt'], 'export.log')
except Exception as exc:
    print('release export failed (evaluation above is unaffected):', exc)
from IPython.display import display, Markdown
display(Markdown((EVAL / 'metrics.md').read_text()))
print('Results in', FT)